<a href="https://colab.research.google.com/github/Mike-Mans/DeepLearning-CS260C-1/blob/main/CS260C_26Spring_hw1_student_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 269C Homework 1

## Deadline: 5/10/2026 at 11:59pm (No Late Submission)


## Outline
- Section 1: Multi-class Image Classification (30pt)
- Section 2: Convolutional Neural Networks (70pt)
  - Section 2.1: Implementing Convolution (30pt)
  - Section 2.2: Training CNN on MNIST and CIFAR-10 (40pt)

## Instructions
- Follow the instructions and fill in the code for the sections marked with `# TODO`.
- **DO NOT** modify the checking/grading cells. Modifying these cells is strictly prohibited and will be treated as an academic integrity violation which will result in 0 score for this assignment and escalation to The Office of Student Conduct at UCLA.

## Submission
- **Execution**: Ensure all cells have been run, and outputs are displayed before submission.
- **File Naming**: Save your completed notebook with outputs as `hw1.ipynb`.
- **Upload**: Submit your `hw1.ipynb` file to Gradescope.

Failure to follow these instructions will result in the autograder failing, which will automatically result in 0 points. **No regrading will be done** for submissions with incorrect file names or formats.

## Section 0: Setup and Preparation

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np

### Load dataset

In [ ]:
# You may change the value of batch size
batch_size = 256

mnist_train_data = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
mnist_test_data = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

mnist_train_dl = torch.utils.data.DataLoader(mnist_train_data, batch_size=batch_size, shuffle=True)
mnist_test_dl = torch.utils.data.DataLoader(mnist_test_data, batch_size=batch_size)

mean = torch.tensor([0.4914, 0.4822, 0.4465])
std = torch.tensor([0.2009, 0.2009, 0.2009])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)])
cifar_train_data = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform)
cifar_test_data = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform)

cifar_train_dl = torch.utils.data.DataLoader(cifar_train_data, batch_size=batch_size, shuffle=True)
cifar_test_dl = torch.utils.data.DataLoader(cifar_test_data, batch_size=batch_size)

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 349kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.24MB/s]
100%|██████████| 170M/170M [02:26<00:00, 1.16MB/s]


## Section 1: Multi-class Image Classification (30pt)

### Model definition

Implement fully-connected neural network (MLP) image classifiers for MNIST and CIFAR-10, respectively.

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, n_features, n_classes, n_neurons=512, activation=nn.ReLU()):
        super().__init__()

        # Define a 3-layer MLP
        self.fc1 = nn.Linear(n_features, n_neurons) # First Hidden Layer
        self.fc2 = nn.Linear(n_neurons, n_neurons) # Second Hidden Layer
        self.fc3 = nn.Linear(n_neurons, n_classes) # Output Layer
        self.activation = activation

    def forward(self, x):
        # Flatten input
        batch_size = x.size(0)
        x = x.view(batch_size, -1) # flatten [B, C, H, W] → [B, C*H*W]

        # Run the forward
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)

        # Return the result/logits
        return x

In [ ]:
n_features_mnist = mnist_train_data.data.shape[1] * mnist_train_data.data.shape[2]
n_classes_mnist = len(mnist_train_data.classes)
n_features_cifar = cifar_train_data.data.shape[1] * cifar_train_data.data.shape[2] * cifar_train_data.data.shape[3]
n_classes_cifar = len(cifar_train_data.classes)

# TODO: Initialize the MLP model for MNIST and CIFAR-10.
model_mnist = MLPModel(n_features_mnist, n_classes_mnist, 512)
model_cifar = MLPModel(n_features_cifar, n_classes_cifar, 1024)

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

if not isinstance(model_mnist, nn.Module):
    raise ValueError("FAILED: model_mnist is not an instance of nn.Module.")
if not hasattr(model_mnist, "fc1") or not isinstance(model_mnist.fc1, nn.Linear):
    raise ValueError("FAILED: model_mnist.fc1 is missing or is not nn.Linear.")
if not hasattr(model_mnist, "fc2") or not isinstance(model_mnist.fc2, nn.Linear):
    raise ValueError("FAILED: model_mnist.fc2 is missing or is not nn.Linear.")
if not hasattr(model_mnist, "fc3") or not isinstance(model_mnist.fc3, nn.Linear):
    raise ValueError("FAILED: model_mnist.fc3 is missing or is not nn.Linear.")

if model_mnist.fc1.out_features != 512:
    raise ValueError(
        f"FAILED: model_mnist.fc1 has incorrect dimensions.\n"
    )
if model_mnist.fc2.in_features != 512 or model_mnist.fc2.out_features != 512:
    raise ValueError(
        f"FAILED: model_mnist.fc2 has incorrect dimensions.\nYour dimensions: ({model_mnist.fc2.in_features}, {model_mnist.fc2.out_features})\nExpected: (512, 512)"
    )
if model_mnist.fc3.in_features != 512:
    raise ValueError(
        f"FAILED: model_mnist.fc3 has incorrect dimensions.\n"
    )
print("Part 1.1 - Structure check passed.")

Part 1.1 - Structure check passed.


### Functions for training and test

In [ ]:
import torch.optim as optim
cuda = torch.device('cuda')
torch.manual_seed(42)

def test(model, test_dl):

    # Evaluate the model on the test dataloader and compute classification accuracy.
    # Return the final accuracy as a float between 0 and 1.
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      for X, y in test_dl:
        X, y = X.cuda(), y.cuda()
        logits = model(X)
        predictions = logits.argmax(dim=1)
        correct += predictions.eq(y).sum().item()
        total += y.size(0)

    accuracy = correct / total
    return accuracy

def train(model, lr, momentum, num_epochs, train_dl, test_dl):
    # Train the model using stochastic gradient descent (SGD).
    # For each epoch:
    #   - Iterate over the training dataloader batch by batch.
    #   - Run a forward pass and Compute the training loss.
    #   - Update model parameters.

    # Define optimizer and loss function
    # Using SGD Optimizer + Cross Entropy Loss
    opt = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    loss_func = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs + 1):
        # Do training on the training set
        model.train()
        for X, y in train_dl:
          X, y = X.cuda(), y.cuda()
          opt.zero_grad(set_to_none=True)
          logits = model(X)
          loss = loss_func(logits, y)
          loss.backward()
          opt.step()
        # Evaluate the model on the test set
        test_accuracy = test(model, test_dl)
        print(f"Test accuracy at epoch {epoch}: {test_accuracy:.4f}")

### Training and Testing for MNIST

In [ ]:
model_mnist.cuda()

#Set hyperparameters
lr = 0.1
momentum = 0.5
num_epochs = 20

train(model_mnist, lr, momentum, num_epochs, mnist_train_dl, mnist_test_dl)
test_mnist_acc = test(model_mnist, mnist_test_dl)
print(f"Final test accuracy on MNIST: {test_mnist_acc:.4f}")

Test accuracy at epoch 1: 0.9128
Test accuracy at epoch 2: 0.9395
Test accuracy at epoch 3: 0.9510
Test accuracy at epoch 4: 0.9576
Test accuracy at epoch 5: 0.9659
Test accuracy at epoch 6: 0.9692
Test accuracy at epoch 7: 0.9717
Test accuracy at epoch 8: 0.9742
Test accuracy at epoch 9: 0.9755
Test accuracy at epoch 10: 0.9750
Test accuracy at epoch 11: 0.9763
Test accuracy at epoch 12: 0.9771
Test accuracy at epoch 13: 0.9788
Test accuracy at epoch 14: 0.9779
Test accuracy at epoch 15: 0.9781
Test accuracy at epoch 16: 0.9775
Test accuracy at epoch 17: 0.9795
Test accuracy at epoch 18: 0.9784
Test accuracy at epoch 19: 0.9793
Test accuracy at epoch 20: 0.9808
Final test accuracy on MNIST: 0.9808


In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

required_accuracy = 0.98

print(f"Your final MNIST test accuracy: {test_mnist_acc:.4f}")
print(f"Required MNIST test accuracy: {required_accuracy:.4f}")

if test_mnist_acc < required_accuracy:
    raise ValueError(
        f"FAILED: MNIST test accuracy is too low.\n"
        f"Your accuracy: {test_mnist_acc:.4f}\n"
        f"Required accuracy: at least {required_accuracy:.4f}\n\n"
        f"Try increasing the number of epochs, adjusting the learning rate, "
        f"or tuning the momentum."
    )

print("Part 1.2 - MNIST accuracy requirement with MLP passed.")

Your final MNIST test accuracy: 0.9808
Required MNIST test accuracy: 0.9800
Part 1.2 - MNIST accuracy requirement with MLP passed.


### Training and Testing for CIFAR-10

In [ ]:
model_cifar.cuda()

#Set hyperparameters
lr = 0.01
momentum = 0.8
num_epochs = 20

train(model_cifar, lr, momentum, num_epochs, cifar_train_dl, cifar_test_dl)
test_cifar_acc = test(model_cifar, cifar_test_dl)
print(f"Final test accuracy on CIFAR-10: {test_cifar_acc:.4f}")

Test accuracy at epoch 1: 0.5765
Test accuracy at epoch 2: 0.5774
Test accuracy at epoch 3: 0.5763
Test accuracy at epoch 4: 0.5819
Test accuracy at epoch 5: 0.5813
Test accuracy at epoch 6: 0.5787
Test accuracy at epoch 7: 0.5787
Test accuracy at epoch 8: 0.5801
Test accuracy at epoch 9: 0.5793
Test accuracy at epoch 10: 0.5793
Test accuracy at epoch 11: 0.5813
Test accuracy at epoch 12: 0.5790
Test accuracy at epoch 13: 0.5802
Test accuracy at epoch 14: 0.5791
Test accuracy at epoch 15: 0.5797
Test accuracy at epoch 16: 0.5809
Test accuracy at epoch 17: 0.5801
Test accuracy at epoch 18: 0.5807
Test accuracy at epoch 19: 0.5804
Test accuracy at epoch 20: 0.5796
Final test accuracy on CIFAR-10: 0.5796


In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

required_accuracy = 0.55

print(f"Your final CIFAR-10 test accuracy: {test_cifar_acc:.4f}")
print(f"Required CIFAR-10 test accuracy: {required_accuracy:.4f}")

if test_cifar_acc < required_accuracy:
    raise ValueError(
        f"FAILED: CIFAR-10 test accuracy is too low.\n"
        f"Your accuracy: {test_cifar_acc:.4f}\n"
        f"Required accuracy: at least {required_accuracy:.4f}\n\n"
        f"Try increasing the number of epochs, adjusting the learning rate, "
        f"or tuning the momentum."
    )

print("Part 1.3 - CIFAR-10 accuracy requirement with MLP passed.")

Your final CIFAR-10 test accuracy: 0.5796
Required CIFAR-10 test accuracy: 0.5500
Part 1.3 - CIFAR-10 accuracy requirement with MLP passed.


## Section 2: Convolutional Neural Networks (70pt)

### Section 2.1 Implementing Convolution Operations (30pts)

In this section, we are going to implement the 2D convolution operation,

Input shape: [batch_size, input_channel, input_height, input_width]

Kernel shape: [output_channel, input_channel, kernel_height, kernel_width]

Stride: an integar

Padding: an integer

In [ ]:
def conv_loop(input, kernel, stride, padding):
    assert type(stride) == int and type(padding) == int
    assert input.shape[0] == 1

    """
    - Implement 2D convolution manually using nested loops.
    - Do not use PyTorch convolution utilities such as nn.Conv2d or F.conv2d.
    """

    # Get input and kernel shape
    _, input_channel, input_height, input_width = input.shape
    output_channel, _, kernel_height, kernel_width = kernel.shape

    # Calculate the output dimensions
    output_height = (input_height + 2 * padding - kernel_height) // stride + 1
    output_width  = (input_width  + 2 * padding - kernel_width)  // stride + 1

    # Initialize output tensor
    output = np.zeros((1, output_channel, output_height, output_width))

    # Pad the input
    padded_input = np.pad(input, ((0,0), (0,0), (padding, padding), (padding, padding)))

    # Perform convolution using nested loops
    # Iterate over the batch, output channels, output height, output width, input channels, kernel height, and kernel width
    for f in range(output_channel):             # which filter
      for i in range(output_height):            # output row
        for j in range(output_width):           # output col
          for c in range(input_channel):        # sum across channels
            for m in range(kernel_height):      # kernel row
              for n in range(kernel_width):     # kernel col
                output[0, f, i, j] += kernel[f, c, m, n] * padded_input[0, c, i*stride + m, j*stride + n]
    # Note: We don't loop over batch_size as the previous assertion states that batch size = 1

    return torch.from_numpy(output).to(torch.float32)

In [ ]:
def conv(input, kernel, stride, padding):
    assert type(stride) == int and type(padding) == int
    assert input.shape[0] == 1

    """
    - Implement convolution with as few explicit for-loops as possible.
    - Do not use F.conv2d, nn.Conv2d, F.unfold, or other high-level convolution utilities.
    - Use tensor indexing and tensor operations to reduce the number of loops.
    """

    # Get input and kernel shape
    _, input_channel, input_height, input_width = input.shape
    output_channel, _, kernel_height, kernel_width = kernel.shape

    # Calculate output dimensions
    output_height = (input_height + 2 * padding - kernel_height) // stride + 1
    output_width  = (input_width  + 2 * padding - kernel_width)  // stride + 1

    # Pad the input
    padded_input = np.pad(input, ((0,0), (0,0), (padding, padding), (padding, padding)))


    # Step 1: Reshape kernel
    kernel_matrix = kernel.reshape(output_channel, input_channel * kernel_height * kernel_width)

    # Step 2: Build patches matrix
    row_indices = (np.arange(output_height) * stride).reshape(-1, 1) + np.arange(kernel_height).reshape(1, -1)
    col_indices = (np.arange(output_width)  * stride).reshape(-1, 1) + np.arange(kernel_width).reshape(1, -1)

    patches_raw = padded_input[0, :, np.ix_(row_indices.flatten(), col_indices.flatten())]
    patches = np.transpose(
        patches_raw.reshape(input_channel, output_height, kernel_height, output_width, kernel_width),
        (0, 2, 4, 1, 3)
    )
    patches = patches.reshape(input_channel * kernel_height * kernel_width, output_height * output_width)

    # Step 3: Matrix multiply
    output = kernel_matrix @ patches

    return torch.from_numpy(output).to(torch.float32)

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

import inspect
import ast
import textwrap

def conv_torch(input, kernel, stride, padding):
    """Reference implementation using PyTorch"""
    return F.conv2d(input, kernel, stride=stride, padding=padding)

def count_for_loops(fn):
    """
    Count the number of `for` loops in a function using AST.
    This counts all Python for-loops appearing in the source code.
    """
    src = inspect.getsource(fn)
    src = textwrap.dedent(src)
    tree = ast.parse(src)

    count = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.For):
            count += 1
    return count

def evaluate_convs(conv_loop_fn, conv_fn):
    # -----------------------------
    # 1. Check correctness
    # -----------------------------
    torch.manual_seed(0)

    input = torch.rand((1, 3, 32, 32))
    kernel_size_all = [1, 3, 5, 7]
    stride_all = [1, 2, 3, 4]
    padding_all = [0, 1, 2, 3, 4]

    conv_loop_correct = True
    conv_correct = True

    for kernel_size in kernel_size_all:
        kernel = torch.rand((4, 3, kernel_size, kernel_size))
        for stride in stride_all:
            for padding in padding_all:
                reference = conv_torch(input, kernel, stride=stride, padding=padding)

                result_loop = conv_loop_fn(input, kernel, stride=stride, padding=padding)
                result = conv_fn(input, kernel, stride=stride, padding=padding)

                if not torch.allclose(reference, result_loop, atol=1e-5):
                    conv_loop_correct = False
                    raise ValueError(f"FAILED: conv_loop is incorrect for kernel_size={kernel_size}, stride={stride}, padding={padding}")

                if not torch.allclose(reference, result, atol=1e-5):
                    conv_correct = False
                    raise ValueError(f"FAILED: conv is incorrect for kernel_size={kernel_size}, stride={stride}, padding={padding}")

    if conv_loop_correct:
        print("Part 2.1.1 - conv_loop implementation passed.")
    if conv_correct:
        print("Part 2.1.2 - Successfully implemented conv.")

    # -----------------------------
    # 2. Count for-loops
    # -----------------------------
    conv_loop_num_loops = count_for_loops(conv_loop_fn)
    conv_num_loops = count_for_loops(conv_fn)

    print(f"conv_loop uses {conv_loop_num_loops} for-loop(s)")
    print(f"conv uses {conv_num_loops} for-loop(s)")

    removed_loops = conv_loop_num_loops - conv_num_loops
    print(f"Part 2.1.2 - conv uses {conv_num_loops} for-loop(s)")

    # -----------------------------
    # 3. Optional scoring message
    # -----------------------------
    if conv_loop_num_loops < 6:
        print("Warning: conv_loop uses fewer than 6 for-loops; check whether it matches the intended straightforward implementation.")

    if removed_loops < 0:
        print("Warning: conv uses more for-loops than conv_loop.")

    score = max(0, min(18, removed_loops * 3))
    print(f"Estimated loop-reduction score for conv: {score}/18")

    if conv_num_loops == 0:
        print("Excellent: conv has no explicit Python for-loops.")

    print("All tests passed.")

evaluate_convs(conv_loop, conv)

### Section 2.2 Training CNN on MNIST and CIFAR-1 (40pts)

It can be slow to train CNN using CPU. You may use GPU runtime on Google Colab (Runtime→Change runtime type). If you decide to use GPU, you need to move PyTorch models and tensors (input and label) to
CUDA by calling .cuda().

You may change your functions for training and test in Section 1.

#### Training and Testing for MNIST

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # TODO: Define your CNN layers.
        pass

    def forward(self, x):
        # TODO: Implement forward pass
        pass

In [ ]:
# TODO: Initialize the CNN model for MNIST classification.
hight_mnist, width_mnist = ...
model_mnist = ...

In [ ]:
""" TODO: Set hyperparameters """
lr =
momentum =
num_epochs =

train(model_mnist, lr, momentum, num_epochs, mnist_train_dl, mnist_test_dl)
test_mnist_acc = test(model_mnist, mnist_test_dl)
print(f"Final test accuracy on MNIST: {test_mnist_acc:.4f}")

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

required_accuracy = 0.99

print(f"Your final MNIST test accuracy: {test_mnist_acc:.4f}")
print(f"Required MNIST test accuracy: {required_accuracy:.4f}")

if test_mnist_acc < required_accuracy:
    raise ValueError(
        f"FAILED: MNIST test accuracy is too low.\n"
        f"Your accuracy: {test_mnist_acc:.4f}\n"
        f"Required accuracy: at least {required_accuracy:.4f}\n\n"
        f"Try increasing the number of epochs, adjusting the learning rate, "
        f"or tuning the momentum."
    )

print("Part 2.2.1 - MNIST accuracy requirement with CNN passed.")

#### Training and Testing for CIFAR-10

In [ ]:
class CIFARCNN(nn.Module):
    def __init__(self, num_classes):
        super(CIFARCNN, self).__init__()

        # TODO: Define your CNN layers.
        pass

    def forward(self, x):
        # TODO: Implement forward pass
        pass

In [ ]:
# TODO: Initialize the CNN model for CIFAR-10 classification.
model_cifar = ...

In [ ]:
""" TODO: Set hyperparameters """
lr =
momentum =
num_epochs =

train(model_cifar, lr, momentum, num_epochs, cifar_train_dl, cifar_test_dl)
test_cifar_acc = test(model_cifar, cifar_test_dl)
print(f"Final test accuracy on CIFAR-10: {test_cifar_acc:.4f}")

In [ ]:
# =====================================================
# DO NOT MODIFY
# =====================================================

required_accuracy = 0.70

print(f"Your final CIFAR-10 test accuracy: {test_cifar_acc:.4f}")
print(f"Required CIFAR-10 test accuracy: {required_accuracy:.4f}")

if test_cifar_acc < required_accuracy:
    raise ValueError(
        f"FAILED: CIFAR-10 test accuracy is too low.\n"
        f"Your accuracy: {test_cifar_acc:.4f}\n"
        f"Required accuracy: at least {required_accuracy:.4f}\n\n"
        f"Try increasing the number of epochs, adjusting the learning rate, "
        f"or tuning the momentum."
    )

print("Part 2.2.2 - CIFAR-10 accuracy requirement with CIFAR-10 passed.")